In [2]:
import pandas as pd
import numpy as np

path = "../data/raw/Mobile_Parts_Wholesale_Dataset.xlsx"
sales = pd.read_excel(path, sheet_name="Sales_Transactions")
inventory = pd.read_excel(path, sheet_name="Inventory")
products = pd.read_excel(path, sheet_name="Products")

sales['order_date'] = pd.to_datetime(sales['order_date'])

print(sales.shape, inventory.shape, products.shape)

(6016, 9) (179, 5) (179, 7)


In [3]:
# Use the last 60 days of history to estimate current demand rate per product
recent_cutoff = sales['order_date'].max() - pd.Timedelta(days=60)
recent_sales = sales[sales['order_date'] >= recent_cutoff]

demand_rate = recent_sales.groupby('product_id').agg(
    qty_sold_last_60d=('quantity', 'sum')
).reset_index()

demand_rate['avg_daily_demand'] = demand_rate['qty_sold_last_60d'] / 60
demand_rate['projected_30d_demand'] = (demand_rate['avg_daily_demand'] * 30).round(1)

print(demand_rate.shape)
demand_rate.head()

(179, 4)


,product_id,qty_sold_last_60d,avg_daily_demand,projected_30d_demand
0,P0001,27,0.450000,13.5
1,P0002,19,0.316667,9.5
2,P0003,60,1.000000,30.0
3,P0004,40,0.666667,20.0
4,P0005,53,0.883333,26.5


In [4]:
alerts = inventory.merge(demand_rate, on='product_id', how='left')
alerts = alerts.merge(products[['product_id', 'product_name', 'brand']], on='product_id', how='left')

# Products with no recent sales get 0 demand (not an error — just slow-moving parts)
alerts['projected_30d_demand'] = alerts['projected_30d_demand'].fillna(0)

conditions = [
    alerts['current_stock_qty'] <= alerts['reorder_level'],
    alerts['projected_30d_demand'] > alerts['current_stock_qty']
]
choices = ["REORDER NOW - Below threshold", "REORDER SOON - Demand exceeds stock"]
alerts['alert_status'] = np.select(conditions, choices, default="Stock OK")

alerts_final = alerts[['product_id', 'product_name', 'brand', 'warehouse_location',
                        'current_stock_qty', 'reorder_level', 'projected_30d_demand', 'alert_status']]

print(alerts_final['alert_status'].value_counts())
alerts_final.sort_values('alert_status').head(15)

alert_status
Stock OK                               144
REORDER NOW - Below threshold           34
REORDER SOON - Demand exceeds stock      1
Name: count, dtype: int64


,product_id,product_name,brand,warehouse_location,current_stock_qty,reorder_level,projected_30d_demand,alert_status
0,P0001,iPhone 11 OLED Display Combo,Apple,Surat Main Warehouse,22,35,13.5,REORDER NOW - Below threshold
43,P0044,Galaxy A32 Speaker Module,Samsung,Ahmedabad Depot,0,30,25.5,REORDER NOW - Below threshold
45,P0046,Galaxy A32 LCD Display Combo,Samsung,Surat Main Warehouse,39,48,25.5,REORDER NOW - Below threshold
50,P0051,Galaxy A52 Camera Module (Front),Samsung,Surat Main Warehouse,23,28,14.5,REORDER NOW - Below threshold
55,P0056,Galaxy M31 Speaker Module,Samsung,Surat Main Warehouse,42,43,25.5,REORDER NOW - Below threshold
64,P0065,Galaxy S21 OLED Display Combo,Samsung,Surat Main Warehouse,10,31,15.5,REORDER NOW - Below threshold
66,P0067,Galaxy S21 Camera Module (Rear),Samsung,Surat Main Warehouse,38,51,19.5,REORDER NOW - Below threshold
80,P0081,Redmi Note 10 Touch Screen Digitizer,Xiaomi,Ahmedabad Depot,14,32,6.5,REORDER NOW - Below threshold
81,P0082,Redmi Note 11 Camera Module (Rear),Xiaomi,Surat Main Warehouse,33,35,13.5,REORDER NOW - Below threshold
84,P0085,Redmi Note 11 Speaker Module,Xiaomi,Ahmedabad Depot,41,43,23.5,REORDER NOW - Below threshold


In [5]:
alerts_final.to_csv("../reports/inventory_alerts.csv", index=False)
print("Saved inventory_alerts.csv")

Saved inventory_alerts.csv
